# Linear Probe Results Analysis

This notebook loads the saved linear-probe evaluation artifacts, validates their shared contract, and visualizes the metrics and confusion matrices. It does not train probes or load fitted estimators: the project intentionally persists reproducible results and provenance rather than serialized scikit-learn objects.

By default it reads the canonical clean Phase 0 artifacts under `outputs/phase0/job-91108/best_loss/probes`. Set `CODY_JEPA_PROBE_DIR` to analyze another probe output directory.

Launch this notebook with `uv run jupyter lab notebooks/linear-probe-results.ipynb`. This project uses uv exclusively: use `uv sync`, `uv add`, `uv remove`, and `uv lock` for the environment, and launch every Python or Jupyter command through `uv run`.

## Imports and artifact location

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/cody_jepa").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the cody-jepa repository root")

REPO_ROOT = find_repo_root()
configured_dir = Path(os.environ.get("CODY_JEPA_PROBE_DIR", "outputs/phase0/job-91108/best_loss/probes")).expanduser()
PROBE_DIR = configured_dir.resolve() if configured_dir.is_absolute() else (REPO_ROOT / configured_dir).resolve()
SUMMARY_PATH = PROBE_DIR / "probe_metrics.csv"
DETAIL_PATH = PROBE_DIR / "probe_metrics.json"
print(f"Repository: {REPO_ROOT}")
print(f"Probe artifacts: {PROBE_DIR}")

## Load and validate

The CSV is the readable scalar summary. The JSON is the canonical full-detail artifact and preserves ordered class labels, confusion matrices, protocol settings, and run provenance. The checks below fail early if the files disagree.

In [ ]:
for artifact in (SUMMARY_PATH, DETAIL_PATH):
    if not artifact.is_file():
        raise FileNotFoundError(f"Missing probe artifact: {artifact}")

summary = pd.read_csv(SUMMARY_PATH)
payload = json.loads(DETAIL_PATH.read_text())
results = payload.get("results")
if not isinstance(results, list) or not results:
    raise ValueError("probe_metrics.json must contain a non-empty results list")

required_summary = [
    "task", "protocol", "feature_source", "train_examples",
    "val_examples", "num_classes", "majority_baseline",
    "accuracy", "balanced_accuracy", "macro_f1",
]
if list(summary.columns) != required_summary:
    raise ValueError(f"Unexpected summary columns: {list(summary.columns)}")
if summary["task"].duplicated().any():
    raise ValueError("probe_metrics.csv contains duplicate tasks")
json_tasks = [str(result["task"]) for result in results]
if summary["task"].astype(str).tolist() != json_tasks:
    raise ValueError("CSV and JSON task order differs")

matrices = {}
class_labels = {}
for csv_row, result in zip(summary.to_dict("records"), results):
    task = str(result["task"])
    labels = [str(label) for label in result.get("class_labels", [])]
    matrix = np.asarray(result.get("confusion_matrix", []))
    expected_classes = int(result["num_classes"])
    if len(labels) != expected_classes or len(set(labels)) != len(labels):
        raise ValueError(f"{task}: class labels are missing or not unique")
    if matrix.shape != (expected_classes, expected_classes):
        raise ValueError(f"{task}: confusion matrix shape {matrix.shape} is invalid")
    if not np.issubdtype(matrix.dtype, np.integer) or (matrix < 0).any():
        raise ValueError(f"{task}: confusion matrix must contain non-negative integers")
    if int(matrix.sum()) != int(result["val_examples"]):
        raise ValueError(f"{task}: confusion-matrix total does not match val_examples")
    measured_accuracy = float(np.trace(matrix) / matrix.sum())
    if not np.isclose(measured_accuracy, float(result["accuracy"])):
        raise ValueError(f"{task}: confusion matrix does not reproduce accuracy")
    for column in required_summary:
        csv_value, json_value = csv_row[column], result[column]
        if isinstance(json_value, (int, float)):
            if not np.isclose(float(csv_value), float(json_value)):
                raise ValueError(f"{task}: CSV and JSON disagree on {column}")
        elif str(csv_value) != str(json_value):
            raise ValueError(f"{task}: CSV and JSON disagree on {column}")
    matrices[task] = matrix
    class_labels[task] = labels

print(f"Validated {len(results)} probe tasks across both artifacts.")

## Provenance and compact metrics

In [ ]:
provenance = {key: value for key, value in payload.items() if key != "results"}
provenance_table = pd.DataFrame(
    [(key, json.dumps(value) if isinstance(value, (dict, list)) else value) for key, value in provenance.items()],
    columns=["field", "value"],
)
display(provenance_table)

metric_columns = ["task", "protocol", "train_examples", "val_examples", "num_classes", "majority_baseline", "accuracy", "balanced_accuracy", "macro_f1"]
display(summary[metric_columns].style.format({
    "majority_baseline": "{:.2%}",
    "accuracy": "{:.2%}",
    "balanced_accuracy": "{:.2%}",
    "macro_f1": "{:.2%}",
}))

## Metric comparison

Accuracy is shown beside the majority-class baseline. Balanced accuracy and macro F1 expose class imbalance that plain accuracy can conceal.

In [ ]:
chart_metrics = ["majority_baseline", "accuracy", "balanced_accuracy", "macro_f1"]
metric_names = ["Majority baseline", "Accuracy", "Balanced accuracy", "Macro F1"]
x = np.arange(len(summary))
width = 0.19
fig, ax = plt.subplots(figsize=(12, 5.5))
for index, (column, label) in enumerate(zip(chart_metrics, metric_names)):
    offset = (index - (len(chart_metrics) - 1) / 2) * width
    ax.bar(x + offset, summary[column], width, label=label)
ax.set_xticks(x, [task.replace("_", "\n") for task in summary["task"]])
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Linear probe performance")
ax.legend(ncol=2)
fig.tight_layout()
plt.show()

## Confusion matrices

Rows are true classes and columns are predicted classes. Each plot is row-normalized, so its color scale represents the fraction of each true class assigned to a predicted class. Small matrices show labels and percentages directly. High-cardinality identity matrices use sampled ticks to remain legible.

In [ ]:
def row_normalize(matrix: np.ndarray) -> np.ndarray:
    row_totals = matrix.sum(axis=1, keepdims=True)
    return np.divide(matrix, row_totals, out=np.zeros_like(matrix, dtype=float), where=row_totals != 0)

def plot_confusion_matrix(task: str, max_ticks: int = 12) -> None:
    matrix = matrices[task]
    labels = class_labels[task]
    normalized = row_normalize(matrix)
    size = len(labels)
    fig_size = (7, 6) if size <= 20 else (9, 7)
    fig, ax = plt.subplots(figsize=fig_size)
    image = ax.imshow(normalized, cmap="Blues", vmin=0, vmax=1, aspect="auto", interpolation="nearest")
    if size <= 20:
        ticks = np.arange(size)
        tick_labels = labels
        for row in range(size):
            for column in range(size):
                value = normalized[row, column]
                ax.text(column, row, f"{matrix[row, column]}\n{value:.1%}", ha="center", va="center", color="white" if value > 0.5 else "black", fontsize=9)
    else:
        ticks = np.unique(np.linspace(0, size - 1, min(max_ticks, size), dtype=int))
        tick_labels = [labels[index] for index in ticks]
    ax.set_xticks(ticks, tick_labels, rotation=45, ha="right")
    ax.set_yticks(ticks, tick_labels)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"{task.replace('_', ' ').title()} — row-normalized confusion matrix")
    fig.colorbar(image, ax=ax, label="Fraction of true class")
    fig.tight_layout()
    plt.show()

for task in json_tasks:
    plot_confusion_matrix(task)

## Per-class recall and largest errors

For many-class identity probes, a distribution is clearer than hundreds of bars. The table beneath each plot lists the largest off-diagonal errors with both class labels.

In [ ]:
def per_class_diagnostics(task: str, top_n: int = 12) -> None:
    matrix = matrices[task]
    labels = class_labels[task]
    normalized = row_normalize(matrix)
    recalls = np.diag(normalized)
    display(Markdown(f"### {task.replace('_', ' ').title()}"))
    diagnostic_summary = pd.DataFrame({
        "classes": [len(labels)],
        "median_recall": [float(np.median(recalls))],
        "mean_recall": [float(np.mean(recalls))],
        "zero_recall_classes": [int(np.count_nonzero(recalls == 0))],
    })
    display(diagnostic_summary.style.format({"median_recall": "{:.2%}", "mean_recall": "{:.2%}"}))
    fig, ax = plt.subplots(figsize=(10, 4))
    if len(labels) <= 20:
        ax.bar(labels, recalls)
        ax.tick_params(axis="x", rotation=45)
        ax.set_ylabel("Recall")
    else:
        ax.hist(recalls, bins=np.linspace(0, 1, 21), edgecolor="white")
        ax.set_xlabel("Per-class recall")
        ax.set_ylabel("Number of classes")
    ax.set_title("Per-class recall")
    fig.tight_layout()
    plt.show()

    off_diagonal = matrix.copy()
    np.fill_diagonal(off_diagonal, 0)
    ranked = np.argsort(off_diagonal, axis=None)[::-1]
    rows = []
    for flat_index in ranked:
        true_index, predicted_index = np.unravel_index(flat_index, off_diagonal.shape)
        count = int(off_diagonal[true_index, predicted_index])
        if count == 0 or len(rows) >= top_n:
            break
        support = int(matrix[true_index].sum())
        rows.append({
            "true_label": labels[true_index],
            "predicted_label": labels[predicted_index],
            "count": count,
            "fraction_of_true_class": count / support if support else 0.0,
        })
    if rows:
        display(pd.DataFrame(rows).style.format({"fraction_of_true_class": "{:.2%}"}))
    else:
        display(Markdown("No off-diagonal errors."))

for task in json_tasks:
    per_class_diagnostics(task)

## Interpretation guardrails

These probes measure how readily each target is recovered from a frozen representation under its stated protocol. Strong performance is evidence of linear recoverability, not causality, clinical validity, or a uniquely disentangled representation. Compare runs only when their feature source, split, probe protocol, and checkpoint provenance are compatible.